## Процессы

### Задание 1

Используя multiprocessing.Pool, реализуйте параллельную обработку списка чисел. Напишите функцию, которая возводит число в квадрат. Создайте пул из 4 процессов и используйте map.

In [15]:
from multiprocessing import Process, Value, Lock
import multiprocessing
import threading
import asyncio
import queue
import random
import time
import os

In [2]:
import multiprocessing

numbers = [1, 2, 3, 4, 5]
# ваш код здесь

def square(x):
    return x * x

with multiprocessing.Pool(processes=4) as pool:
    results = pool.map(square, numbers)

print(results)



[1, 4, 9, 16, 25]


### Задание 2

Создайте общую переменную Value и защитите её с помощью Lock, чтобы два процесса, одновременно прибавляя единицу 1000 раз, выдали в итоге ровно 2000.

In [3]:
def increment(shared_val, lock):
  for _ in range(1000):
    with lock:
      shared_val.value += 1

counter = Value('i', 0)
lock = Lock()

p1 = Process(target=increment, args=(counter, lock))
p2 = Process(target=increment, args=(counter, lock))

p1.start()
p2.start()

p1.join()
p2.join()

# ваш код здесь
print(f"Итог: {counter.value}")

Итог: 2000


### Задание 3

Реализуйте передачу строки через Queue. Дочерний процесс должен получить строку и напечатать её вместе со своим PID (используйте os.getpid()).

In [4]:
import os

def worker(q):
  msg = q.get()
  print(f"[PID {os.getpid()}] Получено сообщение: {msg}")

queue = multiprocessing.Queue()
p = multiprocessing.Process(target=worker, args=(queue,))
p.start()
# Отправьте сообщение в очередь
# ваш код здесь
queue.put("Привет из главного процесса!")

p.join()

[PID 1403] Получено сообщение: Привет из главного процесса!


### Задание 4

Условие:

Создайте два процесса, которые «общаются» друг с другом.

- Процесс sender отправляет число в Pipe.

- Процесс receiver получает это число, возводит его в квадрат и отправляет результат обратно в этот же Pipe.

- Процесс sender получает ответ и выводит его на экран.

In [5]:
def worker_receiver(conn):
  num = conn.recv()
  result = num ** 2
  conn.send(result)

# ваш код здесь
parent_conn, child_conn = multiprocessing.Pipe()

p = multiprocessing.Process(target=worker_receiver, args=(child_conn,))
p.start()

parent_conn.send(7)

result = parent_conn.recv()

print(f"Получен результат: {result}")

p.join()

Получен результат: 49


### Задание 5

Используйте Manager, чтобы создать общий словарь. Каждый процесс должен записать в него свое имя (ключ) и свой PID (значение).

In [6]:
def register_process(shared_dict):
    name = multiprocessing.current_process().name
    pid = os.getpid()

    shared_dict[name] = (pid, "успешно зарегистрирован")

manager = multiprocessing.Manager()
shared_dict = manager.dict()

# ваш код здесь

processes = []

for i in range(5):
    p = multiprocessing.Process(
        target=register_process,
        args=(shared_dict,),
        name=f"Process-{i}"
    )
    processes.append(p)
    p.start()

for p in processes:
    p.join()

for name, info in shared_dict.items():
  print(f"Процесс {name} (PID {info[0]}) записал: {info[1]}")

Процесс Process-0 (PID 1707) записал: успешно зарегистрирован
Процесс Process-1 (PID 1711) записал: успешно зарегистрирован
Процесс Process-2 (PID 1712) записал: успешно зарегистрирован
Процесс Process-3 (PID 1716) записал: успешно зарегистрирован
Процесс Process-4 (PID 1721) записал: успешно зарегистрирован


##  Потоки

In [7]:
import random
import queue

## Задание 1

Представьте, что этот код запущен в Python 3.13+ с флагом --disable-gil (free-threading)

Задача: Измените код внутри функции increment так, чтобы возникла ситуация Race Condition.

При запуске 10 потоков итоговое значение counter должно быть меньше 1 000 000 (c GIL будет 1 000 000).

Подсказка: сделайте операцию инкремента неатомарной, разбив её на чтение и запись.

In [8]:
counter = 0

def increment():
  global counter
  for _ in range(100000):
    # ваш код здесь
    temp = counter
    temp += 1
    counter = temp

threads = [threading.Thread(target=increment) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Итоговый счетчик: {counter}")

Итоговый счетчик: 1000000


### Задание 2

Допишите логику синхронизации в функции worker.
Все 5 потоков должны завершить "Фазу 1" и только потом одновременно начать "Фазу 2".

Использовать threading.Barrier запрещено.

Используйте Lock и Event.

In [9]:
def worker(worker_id, info):
    print(f"[Поток {worker_id}] Начинает Фазу 1...")
    time.sleep(random.uniform(0.5, 1.5))
    print(f"[Поток {worker_id}] Завершил Фазу 1")

    with info["lock"]:
        info["counter"] += 1

        if info["counter"] == 5:
            info["event"].set()

    info["event"].wait()

    print(f"[Поток {worker_id}] >>> ПЕРЕШЕЛ К ФАЗЕ 2")

shared_data = {
    "counter": 0,
    "lock": threading.Lock(),
    "event": threading.Event()
}

threads = []
for i in range(5):
    t = threading.Thread(target=worker, args=(i, shared_data))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("\nВсе потоки успешно прошли барьер.")

[Поток 0] Начинает Фазу 1...
[Поток 1] Начинает Фазу 1...
[Поток 2] Начинает Фазу 1...
[Поток 3] Начинает Фазу 1...
[Поток 4] Начинает Фазу 1...
[Поток 0] Завершил Фазу 1
[Поток 4] Завершил Фазу 1
[Поток 3] Завершил Фазу 1
[Поток 2] Завершил Фазу 1
[Поток 1] Завершил Фазу 1
[Поток 1] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 4] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 2] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 3] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 0] >>> ПЕРЕШЕЛ К ФАЗЕ 2

Все потоки успешно прошли барьер.


### Задание 3

Задача: Реализуйте функцию воркера, которая обрабатывает задачи из очереди.

Поток должен корректно завершиться (выйти из цикла) в двух ситуациях:
1. Если в очередь пришел специальный сигнал остановки — None (poison pill).
2. Если переданный Event (stop_event) перешел в состояние True.

ВАЖНО: Поток не должен блокироваться на q.get() вечно, если работа должна быть прекращена.

In [10]:
def smart_worker(q, stop_event):
    while True:
        # ваш код здесь

        # Подсказка: используйте q.get(timeout=...) чтобы поток мог просыпаться
        # и проверять состояние stop_event.
        if stop_event.is_set():
            print("Получен сигнал остановки через Event")
            break

        try:
            task = q.get(timeout=0.5)

            if task is None:
                print("Получен poison pill")
                break

            print(f"Обрабатываю задачу: {task}")

        except queue.Empty:
            continue

task_queue = queue.Queue()
stop_signal = threading.Event()

worker_thread = threading.Thread(target=smart_worker, args=(task_queue, stop_signal))
worker_thread.start()

task_queue.put("Обработать данные пользователя")
time.sleep(1)

print("Подаем сигнал остановки через Event...")
stop_signal.set()
worker_thread.join()

print("Программа успешно завершена.")

Обрабатываю задачу: Обработать данные пользователя
Подаем сигнал остановки через Event...
Получен сигнал остановки через Event
Программа успешно завершена.


### Задание 4

Задача: Ниже представлен код, который гарантированно приводит к зависанию (Deadlock).

Поток 1 захватывает замок A и ждет B. Поток 2 захватывает замок B и ждет A.

Исправьте одну из функций так, чтобы дедлока не возникало.

Соблюдайте правило: все потоки должны захватывать одни и те же замки в одинаковом порядке.

In [11]:
lock_a = threading.Lock()
lock_b = threading.Lock()

def process_one():
    with lock_a:
        print("[P1] Захватил Lock A, думаю...")
        time.sleep(0.5)
        print("[P1] Пытаюсь захватить Lock B...")
        with lock_b:
            print("[P1] Успех! Выполнил задачу.")

def process_two():
    with lock_a:
        print("[P2] Захватил Lock B, думаю...")
        time.sleep(0.5)
        print("[P2] Пытаюсь захватить Lock A...")
        with lock_b:
            print("[P2] Успех! Выполнил задачу.")

t1 = threading.Thread(target=process_one)
t2 = threading.Thread(target=process_two)

t1.start()
t2.start()

t1.join()
t2.join()
print("Программа завершена без дедлока!")

[P1] Захватил Lock A, думаю...
[P1] Пытаюсь захватить Lock B...
[P1] Успех! Выполнил задачу.
[P2] Захватил Lock B, думаю...
[P2] Пытаюсь захватить Lock A...
[P2] Успех! Выполнил задачу.
Программа завершена без дедлока!


### Задание 5

Задача: Реализуйте паттерн Singleton (Одиночка) так, чтобы при вызове DatabaseConnection() из 100 разных потоков, все они получили ссылку на один и тот же объект в памяти.

ВАЖНО: Используйте механизм Double-Checked Locking для оптимизации производительности.

In [12]:
class DatabaseConnection:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        # ваш код здесь
        if cls._instance is None:

            with cls._lock:

                # вторая проверка внутри lock
                if cls._instance is None:
                    cls._instance = super().__new__(cls)

        return cls._instance

def test_singleton():
    obj = DatabaseConnection()
    print(f"Поток {threading.current_thread().name} получил объект ID: {id(obj)}")

threads = []
for i in range(10):
    t = threading.Thread(target=test_singleton, name=f"T-{i}")
    threads.append(t)
    t.start()

for t in threads:
    t.join()

Поток T-0 получил объект ID: 136539991765600
Поток T-1 получил объект ID: 136539991765600
Поток T-2 получил объект ID: 136539991765600
Поток T-3 получил объект ID: 136539991765600
Поток T-4 получил объект ID: 136539991765600
Поток T-5 получил объект ID: 136539991765600
Поток T-6 получил объект ID: 136539991765600
Поток T-7 получил объект ID: 136539991765600
Поток T-8 получил объект ID: 136539991765600
Поток T-9 получил объект ID: 136539991765600


## asyncio

In [13]:
import asyncio
import time
import nest_asyncio
nest_asyncio.apply()

### Задание 1

1. Напишите корутину fetch_data(id, delay), которая имитирует загрузку данных: выводит сообщение о старте, спит delay секунд (асинхронно!) и возвращает строку f"Data-{id}".
2. Напишите корутину main(), которая запускает 3 таких загрузки конкурентно с задержками 1, 2 и 3 секунды соответственно.
3. Выведите итоговый список результатов и общее время выполнения (оно должно быть около 3 сек, а не 6).

In [17]:
async def fetch_data(data_id, delay):
    # ваш код здесь
    print(f"Начинаю загрузку Data-{data_id}")

    await asyncio.sleep(delay)

    print(f"Загрузка Data-{data_id} завершена")

    return f"Data-{data_id}"

async def main():
    start_time = time.perf_counter()

    # ваш код здесь
    results = await asyncio.gather(
        fetch_data(1, 1),
        fetch_data(2, 2),
        fetch_data(3, 3),
    )

    end_time = time.perf_counter()
    print(f"Результаты: {results}")
    print(f"Затрачено времени: {end_time - start_time:.2f} сек")

asyncio.run(main())

Начинаю загрузку Data-1
Начинаю загрузку Data-2
Начинаю загрузку Data-3
Загрузка Data-1 завершена
Загрузка Data-2 завершена
Загрузка Data-3 завершена
Результаты: ['Data-1', 'Data-2', 'Data-3']
Затрачено времени: 3.00 сек


### Задание 2

1. Напишите корутину slow_api_call(), которая спит 5 секунд и возвращает "Success".
2. В main() вызовите эту корутину, но ограничьте её выполнение 2 секундами с помощью asyncio.wait_for.
3. Обработайте исключение asyncio.TimeoutError, чтобы программа не падала, а выводила "API запрос занял слишком много времени!".

In [18]:
async def slow_api_call():
    # ваш код здесь
    await asyncio.sleep(5)

    return "Success"

async def main():
    # ваш код здесь (try/except блок)
    try:
        result = await asyncio.wait_for(
            slow_api_call(),
            timeout=2
        )

        print(result)

    except asyncio.TimeoutError:
        print("API запрос занял слишком много времени!")

asyncio.run(main())

API запрос занял слишком много времени!


### Задание 3

Представьте, что вам нужно проверить доступность 20 сайтов.
Если запустить все 20 запросов одновременно, сервер может расценить это как атаку.

1. Напишите корутину fetch_url(url, semaphore), которая:
    - Использует семафор для ограничения входа (одновременно не более 3-х).
    - Имитирует запрос (asyncio.sleep от 1 до 3 сек).
    - Выводит сообщение: "[Запрос] Проверка {url} началась".
    - После "ответа" выводит: "[Готово] {url} проверен".
2. В main() создайте список из 20 условных URL (например, site_1, site_2...) и запустите их конкурентно, но с ограничением семафора.

In [19]:
import random

async def fetch_url(url, semaphore):
    # ваш код здесь
    # Используйте 'async with semaphore:', чтобы ограничить количество запросов
    async with semaphore:

        print(f"[Запрос] Проверка {url} началась")

        await asyncio.sleep(random.uniform(1, 3))

        print(f"[Готово] {url} проверен")

async def main():
    sem = asyncio.Semaphore(3)
    urls = [f"https://site_{i}.com" for i in range(1, 21)]

    # ваш код здесь: создайте список задач и запустите их через gather
    tasks = [
        fetch_url(url, sem)
        for url in urls
    ]

    await asyncio.gather(*tasks)

asyncio.run(main())

[Запрос] Проверка https://site_1.com началась
[Запрос] Проверка https://site_2.com началась
[Запрос] Проверка https://site_3.com началась
[Готово] https://site_2.com проверен
[Запрос] Проверка https://site_4.com началась
[Готово] https://site_3.com проверен
[Запрос] Проверка https://site_5.com началась
[Готово] https://site_1.com проверен
[Запрос] Проверка https://site_6.com началась
[Готово] https://site_4.com проверен
[Запрос] Проверка https://site_7.com началась
[Готово] https://site_6.com проверен
[Запрос] Проверка https://site_8.com началась
[Готово] https://site_5.com проверен
[Запрос] Проверка https://site_9.com началась
[Готово] https://site_7.com проверен
[Запрос] Проверка https://site_10.com началась
[Готово] https://site_8.com проверен
[Запрос] Проверка https://site_11.com началась
[Готово] https://site_9.com проверен
[Запрос] Проверка https://site_12.com началась
[Готово] https://site_10.com проверен
[Запрос] Проверка https://site_13.com началась
[Готово] https://site_11.co